# DRAM-V Pfam hit counts

Compare per-gene Pfam hit counts from DRAM-V, CheckAMG, and stock HMMER on the benchmark genes, against the same Pfam release. DRAM-V searches Pfam with `mmseqs search -k 5 -s 7` and no E-value filter ([`mag_annotator/annotate_bins.py::run_mmseqs_profile_search`](https://github.com/WrightonLabCSU/DRAM/blob/master/mag_annotator/annotate_bins.py#L215)), and `DRAM-v.py distill` turns the resulting hits into AMG calls in `amg_summary.tsv`.

Scripts used here are in `accessory_scripts/`. Large intermediate files are written to `MAIN_DIR` outside the repository, and smaller tables to `tables/DRAMV_pfam/`.

In [1]:
! pip install polars --quiet

In [2]:
from pathlib import Path
ROOT_DIR = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark")
MAIN_DIR = ROOT_DIR.joinpath("dramv_pfam_analysis")
MAIN_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR = Path("./accessory_scripts")
OUTPUT_TABLES_DIR = Path("./tables/DRAMV_pfam")
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
import subprocess
import os
os.environ["POLARS_MAX_THREADS"] = str(100)
import polars as pl
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(10)

polars.config.Config

## Get per-gene Pfam counts across DRAM-V runs

- **DRAM-V `pfam_hits`**: parsed from `annotations.tsv` from `DRAM-V.py annotate` per sample, regex `PF\d{5}` on the `pfam_hits` column
- **CheckAMG**: parsed from `wdir/hmm_results.parquet` filtered to `db == 'Pfam'`, with both `keep==true` and the pre-filter count (any `keep`)

In [4]:
DRAMV_OUTPUT_DIR = ROOT_DIR.joinpath("dramv_outputs")
DRAMV_RAW_ANNOTS_DIR = ROOT_DIR.joinpath("dramv_annotations")
DRAMV_GENES_DIR = ROOT_DIR.joinpath("dramv_genes_reformatted")
CHECKAMG_OUTPUT_DIR = ROOT_DIR.joinpath("checkamg_annotate_v1.1_outputs")

In [5]:
import re
PFAM_RE = re.compile(r"(PF\d{5})(?:\.\d+)?")

In [6]:
import gzip

def discover_samples() -> list[tuple[str, str]]:
    samples = []
    for subset in ["metagenomes", "viromes", "complete_virus_genomes"]:
        subset_dir = DRAMV_OUTPUT_DIR / subset
        if not subset_dir.is_dir():
            continue
        for s in sorted(os.listdir(subset_dir)):
            if (subset_dir / s / "annotations.tsv.gz").is_file():
                samples.append((subset, s))
    return samples

def environment_of(sample: str) -> str:
    low = sample.lower()
    if "freshwater" in low or "marine" in low:
        return "freshwater" if "freshwater" in low else "marine"
    if "human_gut" in low or "virus_genomes_gut" in low:
        return "gut"
    if "soil" in low or "virus_genomes_soil" in low:
        return "soil"
    return "other"

def seq_type_of(subset: str) -> str:
    return {
        "metagenomes": "metagenome",
        "viromes": "virome",
        "complete_virus_genomes": "viral_genome",
    }.get(subset, subset)

def parse_dramv_sample(subset: str, sample: str) -> pl.DataFrame:
    path = DRAMV_OUTPUT_DIR / subset / sample / "annotations.tsv.gz"
    print(f"[DRAM-V] {subset}/{sample} parsing {path}")

    with gzip.open(path, "rt") as fh:
        header = fh.readline().rstrip("\n").split("\t")
    # gene column is the first (unnamed in header, index 0)
    if "pfam_hits" not in header:
        raise RuntimeError(f"pfam_hits column missing in {path}")

    # The first column has an empty header, which polars renames
    df = pl.read_csv(
        path,
        separator="\t",
        has_header=True,
        schema_overrides={"pfam_hits": pl.Utf8},
        infer_schema_length=1000,
        null_values=["", "NA"],
    )
    # The gene id column is the first, polars parses empty header as blank
    gene_col = df.columns[0]
    df = df.select([gene_col, "pfam_hits"]).rename({gene_col: "old_gene"})

    df = df.with_columns(
        pl.col("pfam_hits")
        .fill_null("")
        .map_elements(lambda s: list({m.group(1) for m in PFAM_RE.finditer(s)}), return_dtype=pl.List(pl.Utf8))
        .alias("dramv_pfam_accessions")
    ).with_columns(pl.col("dramv_pfam_accessions").list.len().alias("dramv_pfam_n"))
    df = df.drop("pfam_hits")
    df = df.with_columns(pl.lit(subset).alias("subset"), pl.lit(sample).alias("sample"))
    return df

def parse_checkamg_sample(subset: str, sample: str) -> pl.DataFrame:
    path = CHECKAMG_OUTPUT_DIR / subset / sample / "wdir" / "hmm_results.parquet"
    if not path.is_file():
        print(f"[CheckAMG] missing {path}")
        return pl.DataFrame(
            {
                "new_gene": pl.Series([], dtype=pl.Utf8),
                "checkamg_pfam_accessions_all": pl.Series([], dtype=pl.List(pl.Utf8)),
                "checkamg_pfam_accessions_keep": pl.Series([], dtype=pl.List(pl.Utf8)),
                "checkamg_pfam_n_all": pl.Series([], dtype=pl.UInt32),
                "checkamg_pfam_n_keep": pl.Series([], dtype=pl.UInt32),
                "subset": pl.Series([], dtype=pl.Utf8),
                "sample": pl.Series([], dtype=pl.Utf8),
            }
        )
    print(f"[CheckAMG] {subset}/{sample} parsing {path}")
    lf = (
        pl.scan_parquet(path)
        .filter(pl.col("db") == "Pfam")
        .select(["sequence", "hmm_id", "keep"])
        .with_columns(pl.col("hmm_id").str.extract(r"(PF\d{5})", 1).alias("pfam_acc"))
    )
    all_df = (
        lf.group_by("sequence")
        .agg(pl.col("pfam_acc").unique().drop_nulls().alias("checkamg_pfam_accessions_all"))
        .with_columns(pl.col("checkamg_pfam_accessions_all").list.len().alias("checkamg_pfam_n_all"))
    )
    keep_df = (
        lf.filter(pl.col("keep"))
        .group_by("sequence")
        .agg(pl.col("pfam_acc").unique().drop_nulls().alias("checkamg_pfam_accessions_keep"))
        .with_columns(pl.col("checkamg_pfam_accessions_keep").list.len().alias("checkamg_pfam_n_keep"))
    )
    df = all_df.join(keep_df, on="sequence", how="left").collect()
    df = df.with_columns(
        pl.col("checkamg_pfam_accessions_keep").fill_null([]),
        pl.col("checkamg_pfam_n_keep").fill_null(0).cast(pl.UInt32),
    )
    df = df.rename({"sequence": "new_gene"})
    df = df.with_columns(pl.lit(subset).alias("subset"), pl.lit(sample).alias("sample"))
    return df

def load_gene_id_mapping() -> pl.DataFrame:
    path = DRAMV_GENES_DIR / "gene_id_mapping.parquet"
    print(f"[mapping] loading {path}")
    m = pl.read_parquet(path).select(
        ["subset", "sample", "old_gene", "new_gene",
         "start", "end", "frame", "scaffold_len_bases"]
    )
    return m

In [7]:
samples = discover_samples()
print(f"Discovered {len(samples)} samples")

Discovered 21 samples


In [8]:
mapping = load_gene_id_mapping()

[mapping] loading /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_genes_reformatted/gene_id_mapping.parquet


In [9]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_sample(subset, sample):
    dramv = parse_dramv_sample(subset, sample)
    chk = parse_checkamg_sample(subset, sample)
    map_sub = mapping.filter(
        (pl.col("subset") == subset) & (pl.col("sample") == sample)
    ).select(["old_gene", "new_gene", "start", "end", "frame"])

    merged = (
        map_sub
        .join(dramv.select(["old_gene", "dramv_pfam_accessions", "dramv_pfam_n"]), on="old_gene", how="left")
        .join(chk.select(["new_gene", "checkamg_pfam_accessions_all", "checkamg_pfam_n_all",
                          "checkamg_pfam_accessions_keep", "checkamg_pfam_n_keep"]),
              on="new_gene", how="left")
        .with_columns(
            pl.col("dramv_pfam_accessions").fill_null([]),
            pl.col("dramv_pfam_n").fill_null(0).cast(pl.UInt32),
            pl.col("checkamg_pfam_accessions_all").fill_null([]),
            pl.col("checkamg_pfam_accessions_keep").fill_null([]),
            pl.col("checkamg_pfam_n_all").fill_null(0).cast(pl.UInt32),
            pl.col("checkamg_pfam_n_keep").fill_null(0).cast(pl.UInt32),
            pl.lit(subset).alias("subset"),
            pl.lit(sample).alias("sample"),
            pl.lit(seq_type_of(subset)).alias("seq_type"),
            pl.lit(environment_of(sample)).alias("environment"),
        )
    )

    merged = merged.with_columns(
        (((pl.col("end") - pl.col("start")).abs() + 1) // 3 - 1).alias("seq_len_aa")
    )

    print(
        f"  merged rows={merged.height} "
        f"dramv_pfam_n mean={merged['dramv_pfam_n'].mean():.2f} "
        f"checkamg_keep mean={merged['checkamg_pfam_n_keep'].mean():.2f}"
    )

    return merged

In [10]:
with ThreadPoolExecutor() as executor:
    futures = {executor.submit(process_sample, subset, sample): (subset, sample)
               for subset, sample in samples}

    per_sample_tables = [None] * len(samples)
    index_map = {(subset, sample): i for i, (subset, sample) in enumerate(samples)}

    for future in as_completed(futures):
        subset, sample = futures[future]
        per_sample_tables[index_map[(subset, sample)]] = future.result()

[DRAM-V] metagenomes/freshwater_Ga0485157_contigs parsing /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_outputs/metagenomes/freshwater_Ga0485157_contigs/annotations.tsv.gz
[DRAM-V] metagenomes/freshwater_Ga0485158_contigs parsing /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_outputs/metagenomes/freshwater_Ga0485158_contigs/annotations.tsv.gz
[DRAM-V] metagenomes/freshwater_Ga0485159_contigs parsing /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_outputs/metagenomes/freshwater_Ga0485159_contigs/annotations.tsv.gz
[DRAM-V] metagenomes/human_gut_SRR9162900_contigs parsing /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_outputs/metagenomes/human_gut_SRR9162900_contigs/annotations.tsv.gz
[DRAM-V] metagenomes/human_gut_SRR9162906_contigs parsing /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_outputs/

[CheckAMG] metagenomes/human_gut_SRR9162906_contigs parsing /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/checkamg_annotate_v1.1_outputs/metagenomes/human_gut_SRR9162906_contigs/wdir/hmm_results.parquet


  merged rows=13751 dramv_pfam_n mean=273.36 checkamg_keep mean=0.45


[CheckAMG] metagenomes/human_gut_SRR9162900_contigs parsing /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/checkamg_annotate_v1.1_outputs/metagenomes/human_gut_SRR9162900_contigs/wdir/hmm_results.parquet


  merged rows=17641 dramv_pfam_n mean=275.65 checkamg_keep mean=0.48


...

  merged rows=452133 dramv_pfam_n mean=267.25 checkamg_keep mean=0.27


[CheckAMG] viromes/freshwater_Ga0485175_contigs parsing /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/checkamg_annotate_v1.1_outputs/viromes/freshwater_Ga0485175_contigs/wdir/hmm_results.parquet


  merged rows=438684 dramv_pfam_n mean=267.38 checkamg_keep mean=0.31


[CheckAMG] metagenomes/soil_T42_15_2_56 parsing /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/checkamg_annotate_v1.1_outputs/metagenomes/soil_T42_15_2_56/wdir/hmm_results.parquet


  merged rows=573398 dramv_pfam_n mean=260.57 checkamg_keep mean=0.07


In [11]:
per_sample_genes_full = pl.concat(per_sample_tables, how="diagonal_relaxed")

In [12]:
per_sample_genes_full

old_gene,new_gene,start,end,frame,dramv_pfam_accessions,dramv_pfam_n,checkamg_pfam_accessions_all,checkamg_pfam_n_all,checkamg_pfam_accessions_keep,checkamg_pfam_n_keep,subset,sample,seq_type,environment,seq_len_aa
str,str,i64,i64,i64,list[str],u32,list[str],u32,list[str],u32,str,str,str,str,i64
"""Ga0485157_0000001-cat_2_1""","""Ga0485157_0000001_1""",1,588,1,"[""PF08637"", ""PF13875"", … ""PF24239""]",292,"[""PF02397""]",1,"[""PF02397""]",1,"""metagenomes""","""freshwater_Ga0485157_contigs""","""metagenome""","""freshwater""",195
"""Ga0485157_0000001-cat_2_2""","""Ga0485157_0000001_2""",626,2155,1,"[""PF24064"", ""PF01885"", … ""PF24740""]",286,"[""PF18105"", ""PF18604"", ""PF24923""]",3,[],0,"""metagenomes""","""freshwater_Ga0485157_contigs""","""metagenome""","""freshwater""",509
"""Ga0485157_0000001-cat_2_3""","""Ga0485157_0000001_3""",2152,3813,1,"[""PF16116"", ""PF23270"", … ""PF14594""]",293,"[""PF11617""]",1,[],0,"""metagenomes""","""freshwater_Ga0485157_contigs""","""metagenome""","""freshwater""",553
"""Ga0485157_0000001-cat_2_4""","""Ga0485157_0000001_4""",3823,5646,1,"[""PF01695"", ""PF04349"", … ""PF08433""]",291,"[""PF05496"", ""PF01434"", … ""PF06068""]",7,[],0,"""metagenomes""","""freshwater_Ga0485157_contigs""","""metagenome""","""freshwater""",607
"""Ga0485157_0000001-cat_2_5""","""Ga0485157_0000001_5""",5702,6685,-1,"[""PF13382"", ""PF20042"", … ""PF16394""]",286,"[""PF25231"", ""PF10110""]",2,[],0,"""metagenomes""","""freshwater_Ga0485157_contigs""","""metagenome""","""freshwater""",327
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""IMGVR_UViG_3300048851_000201_3300048851_Ga0496060_000201_40082-130131-cat_1_73""","""IMGVR_UViG_3300048851_000201_3300048851_Ga0496060_000201_40082-130131_73""",39528,42152,1,"[""PF08637"", ""PF04348"", … ""PF08433""]",287,"[""PF18668"", ""PF13884"", ""PF12219""]",3,[],0,"""complete_virus_genomes""","""virus_genomes_soil""","""viral_genome""","""soil""",874
"""IMGVR_UViG_3300048851_000201_3300048851_Ga0496060_000201_40082-130131-cat_1_74""","""IMGVR_UViG_3300048851_000201_3300048851_Ga0496060_000201_40082-130131_74""",42166,43284,-1,"[""PF20428"", ""PF04280"", … ""PF23415""]",285,"[""PF01757""]",1,"[""PF01757""]",1,"""complete_virus_genomes""","""virus_genomes_soil""","""viral_genome""","""soil""",372
"""IMGVR_UViG_3300048851_000201_3300048851_Ga0496060_000201_40082-130131-cat_1_75""","""IMGVR_UViG_3300048851_000201_3300048851_Ga0496060_000201_40082-130131_75""",43563,43733,-1,"[""PF04489"", ""PF02448"", … ""PF11677""]",112,[],0,[],0,"""complete_virus_genomes""","""virus_genomes_soil""","""viral_genome""","""soil""",56


In [13]:
PER_SAMPLE_GENES_PARQUET = MAIN_DIR / "dramv_per_gene_pfam_counts.parquet"

In [14]:
per_sample_genes_full.write_parquet(PER_SAMPLE_GENES_PARQUET)

In [15]:
with pl.Config(tbl_width_chars=600):
    print(
        "Summary per seq_type:\n"
        + str(
            per_sample_genes_full.group_by("seq_type").agg(
                pl.len().alias("n_genes"),
                pl.col("dramv_pfam_n").mean().alias("dramv_mean"),
                pl.col("dramv_pfam_n").quantile(0.95).alias("dramv_q95"),
                pl.col("dramv_pfam_n").max().alias("dramv_max"),
                pl.col("checkamg_pfam_n_keep").mean().alias("chk_keep_mean"),
                pl.col("checkamg_pfam_n_keep").quantile(0.95).alias("chk_keep_q95"),
                pl.col("checkamg_pfam_n_keep").max().alias("chk_keep_max"),
            )
        )
    )

Summary per seq_type:
shape: (3, 8)
┌──────────────┬─────────┬────────────┬───────────┬───────────┬───────────────┬──────────────┬──────────────┐
│ seq_type     ┆ n_genes ┆ dramv_mean ┆ dramv_q95 ┆ dramv_max ┆ chk_keep_mean ┆ chk_keep_q95 ┆ chk_keep_max │
│ ---          ┆ ---     ┆ ---        ┆ ---       ┆ ---       ┆ ---           ┆ ---          ┆ ---          │
│ str          ┆ u64     ┆ f64        ┆ f64       ┆ u32       ┆ f64           ┆ f64          ┆ u32          │
╞══════════════╪═════════╪════════════╪═══════════╪═══════════╪═══════════════╪══════════════╪══════════════╡
│ virome       ┆ 2327154 ┆ 269.915618 ┆ 295.0     ┆ 300       ┆ 0.352082      ┆ 1.0          ┆ 47           │
│ metagenome   ┆ 2128238 ┆ 265.586123 ┆ 295.0     ┆ 300       ┆ 0.209625      ┆ 1.0          ┆ 29           │
│ viral_genome ┆ 200893  ┆ 267.35396  ┆ 295.0     ┆ 300       ┆ 0.161653      ┆ 1.0          ┆ 8            │
└──────────────┴─────────┴────────────┴───────────┴───────────┴───────────────┴─────

Per-gene counts in long format for plotting.

In [16]:
seq_type_label = {
    "metagenome": "Mixed metagenomes",
    "virome": "Viromes",
    "viral_genome": "Viral genomes",
}

per_tool_genes_plotting = (
    per_sample_genes_full
    .with_columns(
        pl.col("seq_type")
        .replace(seq_type_label)
        .cast(pl.Categorical)
    )
    .select([
        "seq_type",
        "dramv_pfam_n",
        "checkamg_pfam_n_all",
        "checkamg_pfam_n_keep"
    ])
    .melt(
        id_vars="seq_type",
        value_vars=[
            "dramv_pfam_n",
            "checkamg_pfam_n_all",
            "checkamg_pfam_n_keep"
        ],
        variable_name="tool",
        value_name="n"
    )
    .with_columns(
        pl.col("tool")
        .replace({
            "dramv_pfam_n": "DRAM-V",
            "checkamg_pfam_n_all": "CheckAMG (all)",
            "checkamg_pfam_n_keep": "CheckAMG (kept)"
        })
        .cast(pl.Categorical)
    )
    .group_by(["seq_type", "tool", "n"])
    .len()
    .rename({"len": "count"})
    .with_columns([
        pl.col("n").cast(pl.UInt16),
        pl.col("count").cast(pl.UInt32)
    ])
    .sort(["seq_type", "tool", "n"])
)

/storage2/scratch/kosmopoulos/tmp/ipykernel_611498/786216673.py:8: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  per_sample_genes_full


In [17]:
per_tool_genes_plotting

seq_type,tool,n,count
cat,cat,u16,u32
"""Mixed metagenomes""","""DRAM-V""",0,114
"""Mixed metagenomes""","""DRAM-V""",1,84
"""Mixed metagenomes""","""DRAM-V""",2,117
"""Mixed metagenomes""","""DRAM-V""",3,104
"""Mixed metagenomes""","""DRAM-V""",4,128
…,…,…,…
"""Viral genomes""","""CheckAMG (kept)""",2,428
"""Viral genomes""","""CheckAMG (kept)""",3,38
"""Viral genomes""","""CheckAMG (kept)""",4,4


In [18]:
PER_TOOL_GENES_TSV = OUTPUT_TABLES_DIR / "dramv_pfam_per_gene_counts.tsv"

In [19]:
per_tool_genes_plotting.write_csv(PER_TOOL_GENES_TSV, separator="\t")

## Single-gene checks

Three checks before the benchmark-wide runs: the over-reporting reproduces on one example gene with stock HMMER, DRAM-V's Pfam release matches CheckAMG's, and a standalone `mmseqs search` outside DRAM-V gives the same hit count.

In [20]:
CHECKAMG_DB_DIR = Path("./CheckAMG_annotate_db_v1.1_20260316")
PFAM_HMM_SRC = CHECKAMG_DB_DIR.joinpath('Pfam-A.hmm')
KEGG_HMM_SRC = CHECKAMG_DB_DIR.joinpath('KEGG.hmm')
FOAM_HMM_SRC = CHECKAMG_DB_DIR.joinpath('FOAM.hmm')

In [21]:
PFAM_DIR = MAIN_DIR / 'pfam_db'
PFAM_DIR.mkdir(parents=True, exist_ok=True)
PFAM_HMM = PFAM_DIR / 'Pfam-A.hmm'
PFAM_HMM.unlink(missing_ok=True)
PFAM_HMM.symlink_to(PFAM_HMM_SRC.resolve())
def _press_ok(hmm):
    return all((hmm.parent / f'{hmm.name}.{ext}').stat().st_size > 0
               for ext in ('h3f', 'h3i', 'h3m', 'h3p')
               if (hmm.parent / f'{hmm.name}.{ext}').exists()) and \
           all((hmm.parent / f'{hmm.name}.{ext}').exists()
               for ext in ('h3f', 'h3i', 'h3m', 'h3p'))

if not _press_ok(PFAM_HMM):
    for ext in ('h3f', 'h3i', 'h3m', 'h3p'):
        (PFAM_HMM.parent / f'{PFAM_HMM.name}.{ext}').unlink(missing_ok=True)
    print('hmmpress', PFAM_HMM)
    subprocess.run(['hmmpress', str(PFAM_HMM)], check=True)
print('Pfam HMM ready at', PFAM_HMM)

Pfam HMM ready at /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_pfam_analysis/pfam_db/Pfam-A.hmm


### Single-gene check on `Ga0485158_0010419_2` (170 aa HNH endonuclease)


In [22]:
TARGET_GENE = 'Ga0485158_0010419_2'
TARGET_SUBSET, TARGET_SAMPLE = 'metagenomes', 'freshwater_Ga0485158_contigs'

In [23]:
ADV_DIR = MAIN_DIR.joinpath("adversarial")
ADV_DIR.mkdir(parents=True, exist_ok=True)
TARGET_FA = ADV_DIR / f'{TARGET_GENE}.faa'

In [24]:
faa = DRAMV_GENES_DIR.joinpath(TARGET_SUBSET, TARGET_SAMPLE, "genes_reformatted.faa")
keep, cur, lines = False, None, []
with open(faa) as fh, open(TARGET_FA, 'w') as out:
    for line in fh:
        if line.startswith('>'):
            if keep:
                out.write(f'>{cur}\n' + ''.join(lines))
                break
            cur = line[1:].split()[0]
            keep = (cur == TARGET_GENE)
            lines = []
        elif keep:
            lines.append(line)
    else:
        if keep:
            out.write(f'>{cur}\n' + ''.join(lines))
print('wrote', TARGET_FA, TARGET_FA.read_text().count('\n'), 'lines')

wrote /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_pfam_analysis/adversarial/Ga0485158_0010419_2.faa 4 lines


In [25]:
TARGET_TBL = ADV_DIR.joinpath(f"{TARGET_GENE}.hmmscan.tblout")

In [26]:
subprocess.run([
    'hmmscan', '--cpu', '4', '--noali',
    '--tblout', str(TARGET_TBL),
    str(PFAM_HMM), str(TARGET_FA)
], check=True, stdout=subprocess.DEVNULL)
hits = [l.split() for l in TARGET_TBL.read_text().splitlines() if not l.startswith('#') and l.strip()]
print(f'stock hmmscan hits for {TARGET_GENE}: {len(hits)}')
for h in hits:
    print('  ', h[0], h[1], 'E=' + h[4], 'score=' + h[5], h[18] if len(h) > 18 else '')

stock hmmscan hits for Ga0485158_0010419_2: 1
   HNH_3 PF13392.13 E=2.1e-08 score=34.3 HNH


**DRAM-V's `pfam_hits` for the same gene** (raw count + first few accessions)


In [27]:
dramv_row = (
    parse_dramv_sample(TARGET_SUBSET, TARGET_SAMPLE)
    .filter(pl.col('old_gene') == 'Ga0485158_0010419-cat_2_2')
)
print('DRAM-V Pfam hit count:', int(dramv_row['dramv_pfam_n'][0]))
print('first 10 accessions  :', dramv_row['dramv_pfam_accessions'][0].to_list()[:10])

[DRAM-V] metagenomes/freshwater_Ga0485158_contigs parsing /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_outputs/metagenomes/freshwater_Ga0485158_contigs/annotations.tsv.gz


DRAM-V Pfam hit count: 289
first 10 accessions  : ['PF08553', 'PF09018', 'PF12436', 'PF05793', 'PF21600', 'PF15576', 'PF21120', 'PF13638', 'PF05694', 'PF17097']


### Pfam version overlap (CheckAMG vs DRAM-V)


In [28]:
DRAM_PFAM_DAT = Path('/storage2/databases/dram-latest/v1.5.0/Pfam-A.hmm.dat.gz')

In [29]:
def pfam_acc_from_hmm(hmm_path):
    accs = set()
    with open(hmm_path, 'rb') as fh:
        for line in fh:
            if line.startswith(b'ACC '):
                m = re.match(rb'ACC\s+(PF\d{5})', line)
                if m:
                    accs.add(m.group(1).decode())
    return accs

def pfam_acc_from_dat(path):
    accs = set()
    with gzip.open(path, 'rt') as fh:
        for line in fh:
            if line.startswith('#=GF AC'):
                m = re.search(r'(PF\d{5})', line)
                if m:
                    accs.add(m.group(1))
    return accs

chk_accs = pfam_acc_from_hmm(PFAM_HMM_SRC)
dram_accs = pfam_acc_from_dat(DRAM_PFAM_DAT)
overlap = chk_accs & dram_accs
print(f'CheckAMG Pfam profiles : {len(chk_accs):>6}')
print(f'DRAM-V   Pfam profiles : {len(dram_accs):>6}')
print(f'Intersection           : {len(overlap):>6} ({len(overlap)/max(len(dram_accs),1)*100:.1f}% of DRAM-V)')
print(f'CheckAMG-only          : {len(chk_accs - dram_accs):>6}')
print(f'DRAM-V-only            : {len(dram_accs - chk_accs):>6}')

CheckAMG Pfam profiles :  30134
DRAM-V   Pfam profiles :  23794
Intersection           :  23723 (99.7% of DRAM-V)
CheckAMG-only          :   6411
DRAM-V-only            :     71


### Standalone mmseqs2 reproduction


In [30]:
DRAM_PFAM_MMSPRO = Path('/storage2/databases/dram-latest/v1.5.0/pfam.mmspro')
MMSEQS_BIN = Path('/storage2/scratch/kosmopoulos/miniconda3/envs/DRAM/bin/mmseqs')

In [31]:
mm_dir = ADV_DIR.joinpath("standalone_mmseqs")
mm_dir.mkdir(parents=True, exist_ok=True)
qdb, rdb, tmp = mm_dir/'q', mm_dir/'r', mm_dir/'tmp'
tmp.mkdir(exist_ok=True)
b6 = mm_dir.joinpath("pfam_output.b6")

if MMSEQS_BIN.is_file() and not b6.exists():
    subprocess.run([str(MMSEQS_BIN), 'createdb', str(TARGET_FA), str(qdb)],
                   check=True, stdout=subprocess.DEVNULL)
    subprocess.run([str(MMSEQS_BIN), 'search', str(qdb), str(DRAM_PFAM_MMSPRO),
                    str(rdb), str(tmp), '-k', '5', '-s', '7', '--threads', '8'],
                   check=True, stdout=subprocess.DEVNULL)
    subprocess.run([str(MMSEQS_BIN), 'convertalis', str(qdb), str(DRAM_PFAM_MMSPRO),
                    str(rdb), str(b6)],
                   check=True, stdout=subprocess.DEVNULL)

if b6.exists():
    rows = [l.split('\t') for l in b6.read_text().splitlines() if l.strip()]
    evalues = sorted(float(r[10]) for r in rows)
    print(f'standalone mmseqs2 hits for {TARGET_GENE}: {len(rows)}')
    if evalues:
        print(f'  E-values: min={evalues[0]:.2e}  median={evalues[len(evalues)//2]:.2e}  max={evalues[-1]:.2e}')
        print(f'  N at E=0: {sum(1 for e in evalues if e == 0.0)}')
else:
    print('mmseqs2 binary not found at', MMSEQS_BIN)

standalone mmseqs2 hits for Ga0485158_0010419_2: 289
  E-values: min=0.00e+00  median=2.37e-251  max=5.12e-05
  N at E=0: 93


## Stock hmmscan on a stratified gene set

Genes from three categories (DRAM-V over-reports, concordant, CheckAMG-only) are searched with stock `hmmscan` at default thresholds and with `--cut_ga`, as a reference independent of both tools.

In [32]:
CONCORDANCE_DIR = MAIN_DIR.joinpath("concordance")
CONCORDANCE_DIR.mkdir(parents=True, exist_ok=True)

In [33]:
N_OVER, N_CONCORDANT, N_CHECKAMG_ONLY = 15, 10, 10
SEED = 42
HMM_CPUS = 16

In [34]:
def stratified_sample(df, predicate, n, label):
    sub = df.filter(predicate)
    if sub.height == 0:
        return None
    groups = sub.group_by(['seq_type', 'environment']).agg(pl.len().alias('n')).sort('seq_type', 'environment')
    per = max(1, n // max(groups.height, 1))
    parts = []
    for row in groups.iter_rows(named=True):
        part = sub.filter(
            (pl.col('seq_type') == row['seq_type']) & (pl.col('environment') == row['environment'])
        ).sample(min(per, int(row['n'])), seed=SEED)
        parts.append(part)
    out = pl.concat(parts, how='diagonal_relaxed')
    if out.height < n:
        rest = sub.join(out.select('new_gene'), on='new_gene', how='anti')
        if rest.height:
            out = pl.concat([out, rest.sample(min(n - out.height, rest.height), seed=SEED)],
                            how='diagonal_relaxed')
    return out.with_columns(pl.lit(label).alias('category'))

over = stratified_sample(
    per_sample_genes_full,
    (pl.col('dramv_pfam_n') > 20) & (pl.col('checkamg_pfam_n_keep') <= 2),
    N_OVER, 'dramv_over')
concordant = stratified_sample(
    per_sample_genes_full,
    (pl.col('checkamg_pfam_n_keep') >= 1)
    & (pl.col('dramv_pfam_accessions').list.set_intersection(
        pl.col('checkamg_pfam_accessions_keep')).list.len() >= 1),
    N_CONCORDANT, 'concordant')
checkamg_only = stratified_sample(
    per_sample_genes_full,
    (pl.col('checkamg_pfam_n_keep') >= 1)
    & (pl.col('dramv_pfam_accessions').list.set_intersection(
        pl.col('checkamg_pfam_accessions_keep')).list.len() == 0),
    N_CHECKAMG_ONLY, 'checkamg_only')

selected = pl.concat([x for x in (over, concordant, checkamg_only) if x is not None],
                            how='diagonal_relaxed')
# Always include the canonical example gene
target_row = per_sample_genes_full.filter(pl.col('new_gene') == TARGET_GENE)
if target_row.height and selected.join(target_row.select('new_gene'),
                                              on='new_gene', how='inner').height == 0:
    selected = pl.concat([target_row.with_columns(pl.lit('dramv_over').alias('category')),
                                 selected], how='diagonal_relaxed')

In [35]:
selected.group_by('category', 'seq_type', 'environment').len().sort(
    'category', 'seq_type', 'environment')

category,seq_type,environment,len
str,str,str,u64
"""checkamg_only""","""metagenome""","""freshwater""",2
"""checkamg_only""","""metagenome""","""gut""",1
"""checkamg_only""","""metagenome""","""soil""",1
"""checkamg_only""","""viral_genome""","""gut""",1
"""checkamg_only""","""viral_genome""","""marine""",1
…,…,…,…
"""dramv_over""","""viral_genome""","""marine""",1
"""dramv_over""","""viral_genome""","""soil""",1
"""dramv_over""","""virome""","""freshwater""",2


In [36]:
CONCORDANCE_PARQUET = CONCORDANCE_DIR.joinpath("selected_genes.parquet")

In [37]:
selected.write_parquet(CONCORDANCE_PARQUET)

In [38]:
def extract_subset_to_fasta(selected, out_fa, gene_col='new_gene'):
    """Pull a small set of gene sequences from the per-sample FAAs."""
    needed = {}
    for row in selected.select(['subset', 'sample', gene_col]).iter_rows(named=True):
        needed.setdefault((row['subset'], row['sample']), set()).add(row[gene_col])
    n_written = 0
    with open(out_fa, 'w') as out:
        for (subset, sample), wanted in needed.items():
            faa = DRAMV_GENES_DIR / subset / sample / 'genes_reformatted.faa'
            if not faa.is_file():
                continue
            with open(faa) as fh:
                cur, lines = None, []
                for line in fh:
                    if line.startswith('>'):
                        if cur in wanted:
                            seq = ''.join(lines).replace('*', '')
                            out.write(f'>{cur}\n' + '\n'.join(seq[i:i+80] for i in range(0, len(seq), 80)) + '\n')
                            n_written += 1
                            wanted.discard(cur)
                        cur = line[1:].split()[0]
                        lines = []
                    else:
                        lines.append(line.strip())
                if cur in wanted:
                    seq = ''.join(lines).replace('*', '')
                    out.write(f'>{cur}\n' + '\n'.join(seq[i:i+80] for i in range(0, len(seq), 80)) + '\n')
                    n_written += 1
    return n_written

In [39]:
CONCORDANCE_FASTA = CONCORDANCE_DIR.joinpath("subset_genes.fasta")
n = extract_subset_to_fasta(selected, CONCORDANCE_FASTA)
print('extracted', n, 'sequences ->', CONCORDANCE_FASTA)

extracted 36 sequences -> /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_pfam_analysis/concordance/subset_genes.fasta


In [40]:
def hmmscan_run(hmm, fa, tblout, cut_ga=False, cpus=HMM_CPUS):
    if Path(tblout).exists() and Path(tblout).stat().st_size > 0:
        print('reusing', tblout)
        return
    cmd = ['hmmscan', '--cpu', str(cpus), '--noali', '--tblout', str(tblout)]
    if cut_ga:
        cmd.append('--cut_ga')
    cmd.extend([str(hmm), str(fa)])
    print(' '.join(cmd))
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL)

def parse_hmmscan_tblout(path):
    """Per-(query, pfam_acc) best hit; column order is target,target_acc,query,..."""
    rows = []
    with open(path) as fh:
        for line in fh:
            if line.startswith('#'):
                continue
            f = line.split()
            if len(f) < 9:
                continue
            rows.append({'query': f[2], 'tacc': f[1],
                         'evalue': float(f[4]), 'score': float(f[5])})
    if not rows:
        return pl.DataFrame(schema={'query': pl.Utf8, 'pfam_acc': pl.Utf8,
                                    'evalue': pl.Float64, 'score': pl.Float64})
    df = pl.from_dicts(rows).with_columns(
        pl.col('tacc').str.extract(r'(PF\d{5})', 1).alias('pfam_acc'))
    return df.group_by(['query', 'pfam_acc']).agg(
        pl.col('evalue').min(), pl.col('score').max())

In [41]:
DEFAULT_TBL = CONCORDANCE_DIR.joinpath("hmmscan_default.tblout")
CUTGA_TBL = CONCORDANCE_DIR.joinpath("hmmscan_cutga.tblout")

In [42]:
hmmscan_run(PFAM_HMM, CONCORDANCE_FASTA, DEFAULT_TBL, cut_ga=False)
hmmscan_run(PFAM_HMM, CONCORDANCE_FASTA, CUTGA_TBL,   cut_ga=True)

reusing /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_pfam_analysis/concordance/hmmscan_default.tblout
reusing /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_pfam_analysis/concordance/hmmscan_cutga.tblout


In [43]:
default_results = parse_hmmscan_tblout(DEFAULT_TBL).rename({'query': 'new_gene'})
cutga_results = parse_hmmscan_tblout(CUTGA_TBL).rename({'query': 'new_gene'})

In [44]:
s1_default_counts = (
    default_results.group_by('new_gene')
    .agg(pl.col('pfam_acc').unique().drop_nulls().alias('hmmscan_default_accs'))
    .with_columns(pl.col('hmmscan_default_accs').list.len().alias('hmmscan_default_n'))
)
s1_cutga_counts = (
    cutga_results.group_by('new_gene')
    .agg(pl.col('pfam_acc').unique().drop_nulls().alias('hmmscan_cutga_accs'))
    .with_columns(pl.col('hmmscan_cutga_accs').list.len().alias('hmmscan_cutga_n'))
)

concordance = (
    selected
    .join(s1_default_counts, on='new_gene', how='left')
    .join(s1_cutga_counts, on='new_gene', how='left')
    .with_columns(
        pl.col('hmmscan_default_n').fill_null(0).cast(pl.UInt32),
        pl.col('hmmscan_cutga_n').fill_null(0).cast(pl.UInt32),
    )
)

In [45]:
concordance

old_gene,new_gene,start,end,frame,dramv_pfam_accessions,dramv_pfam_n,checkamg_pfam_accessions_all,checkamg_pfam_n_all,checkamg_pfam_accessions_keep,checkamg_pfam_n_keep,subset,sample,seq_type,environment,seq_len_aa,category,hmmscan_default_accs,hmmscan_default_n,hmmscan_cutga_accs,hmmscan_cutga_n
str,str,i64,i64,i64,list[str],u32,list[str],u32,list[str],u32,str,str,str,str,i64,str,list[str],u32,list[str],u32
"""Ga0485158_0010419-cat_2_2""","""Ga0485158_0010419_2""",2445,2957,-1,"[""PF08553"", ""PF09018"", … ""PF03546""]",289,"[""PF13392""]",1,[],0,"""metagenomes""","""freshwater_Ga0485158_contigs""","""metagenome""","""freshwater""",170,"""dramv_over""","[""PF13392""]",1,"[""PF13392""]",1
"""Ga0485157_0043281-cat_3_4""","""Ga0485157_0043281_4""",2193,2687,-1,"[""PF04489"", ""PF10864"", … ""PF06589""]",289,"[""PF11783""]",1,"[""PF11783""]",1,"""metagenomes""","""freshwater_Ga0485157_contigs""","""metagenome""","""freshwater""",164,"""dramv_over""","[""PF11783""]",1,"[""PF11783""]",1
"""SRR9162900_k127_1324-cat_3_2""","""SRR9162900_k127_1324_2""",2609,4255,1,"[""PF06032"", ""PF06755"", … ""PF17312""]",289,[],0,[],0,"""metagenomes""","""human_gut_SRR9162900_contigs""","""metagenome""","""gut""",548,"""dramv_over""","[""PF07550"", ""PF20445"", … ""PF13364""]",10,"[""PF03629""]",1
"""scaffold_17261_c1-cat_2_2""","""scaffold_17261_c1_2""",1063,1338,1,"[""PF15604"", ""PF14478"", … ""PF03153""]",293,[],0,[],0,"""metagenomes""","""soil_T42_15_1_55""","""metagenome""","""soil""",91,"""dramv_over""",null,0,null,0
"""IMGVR_UViG_3300045988_090184_3300045988_Ga0495776_166993-cat_1_43""","""IMGVR_UViG_3300045988_090184_3300045988_Ga0495776_166993_43""",50774,51346,-1,"[""PF12988"", ""PF08299"", … ""PF17784""]",286,[],0,[],0,"""complete_virus_genomes""","""virus_genomes_gut""","""viral_genome""","""gut""",190,"""dramv_over""","[""PF08774""]",1,null,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""IMGVR_UViG_3300037193_002017_3300037193_Ga0226660_0497261-cat_1_17""","""IMGVR_UViG_3300037193_002017_3300037193_Ga0226660_0497261_17""",13106,15544,1,"[""PF03310"", ""PF06603"", … ""PF23092""]",290,"[""PF05048"", ""PF13229"", ""PF12708""]",3,"[""PF13229""]",1,"""complete_virus_genomes""","""virus_genomes_soil""","""viral_genome""","""soil""",812,"""checkamg_only""","[""PF05048"", ""PF12708"", ""PF13229""]",3,"[""PF05048"", ""PF13229"", ""PF12708""]",3
"""Ga0485173_0011106-cat_3_8""","""Ga0485173_0011106_8""",5481,5975,1,"[""PF04280"", ""PF07899"", … ""PF23853""]",294,"[""PF20789""]",1,"[""PF20789""]",1,"""viromes""","""freshwater_Ga0485173_contigs""","""virome""","""freshwater""",164,"""checkamg_only""","[""PF20789""]",1,"[""PF20789""]",1
"""SRR9161506_SRR9161507_k127_67628-cat_3_3""","""SRR9161506_SRR9161507_k127_67628_3""",1998,3074,1,"[""PF13691"", ""PF03294"", … ""PF11677""]",286,"[""PF01757""]",1,"[""PF01757""]",1,"""viromes""","""human_gut_SRR9161506_contigs""","""virome""","""gut""",358,"""checkamg_only""","[""PF01757""]",1,"[""PF01757""]",1


In [46]:
concordance.group_by('category').agg(
    pl.len().alias('n'),
    pl.col('dramv_pfam_n').mean().alias('dramv_mean'),
    pl.col('checkamg_pfam_n_keep').mean().alias('checkamg_keep_mean'),
    pl.col('hmmscan_default_n').mean().alias('hmmscan_default_mean'),
    pl.col('hmmscan_cutga_n').mean().alias('hmmscan_cutga_mean'),
).sort('category')

category,n,dramv_mean,checkamg_keep_mean,hmmscan_default_mean,hmmscan_cutga_mean
str,u64,f64,f64,f64,f64
"""checkamg_only""",10,290.7,1.0,5.1,1.6
"""concordant""",10,288.1,1.1,3.9,1.5
"""dramv_over""",16,263.0,0.25,2.3125,0.5


In [47]:
OUTPUT_CONCORDANCE_PARQUET = OUTPUT_TABLES_DIR.joinpath("dramv_pfam_concordance.parquet")

In [48]:
concordance.write_parquet(OUTPUT_CONCORDANCE_PARQUET)

## Unified stock hmmsearch over the benchmark gene set

Stock HMMER hit counts for every gene DRAM-V flagged as carrying a Pfam-based AMG call, plus a stratified random sample of 2,000 genes for the distribution comparison. `hmmsearch` is used instead of `hmmscan` because it is much faster for many sequences against many HMMs, and the input FASTA is split into chunks that run in parallel.

**This step takes about 2-3 h on about 2M sequences**, so the cell skips the search if the merged tblouts exist.

In [49]:
UNIFIED_DIR = MAIN_DIR.joinpath("unified_hmmsearch")
CHUNKS_DIR  = UNIFIED_DIR.joinpath("chunks")
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

In [50]:
SAMPLE_N = 2000
N_CHUNKS = 32
INNER_CPUS = 4

In [51]:
df_stratified = per_sample_genes_full.with_columns(
    pl.when(pl.col('dramv_pfam_n') == 0).then(pl.lit('0'))
    .when(pl.col('dramv_pfam_n').is_between(1, 5)).then(pl.lit('1-5'))
    .when(pl.col('dramv_pfam_n').is_between(6, 20)).then(pl.lit('6-20'))
    .when(pl.col('dramv_pfam_n').is_between(21, 100)).then(pl.lit('21-100'))
    .otherwise(pl.lit('100+')).alias('dramv_bucket')
)
groups = df_stratified.group_by(['seq_type', 'environment', 'dramv_bucket']).len()
per = max(10, SAMPLE_N // max(groups.height, 1))
parts = []
for row in groups.iter_rows(named=True):
    sub = df_stratified.filter(
        (pl.col('seq_type') == row['seq_type']) &
        (pl.col('environment') == row['environment']) &
        (pl.col('dramv_bucket') == row['dramv_bucket']))
    take = min(per, sub.height)
    if take:
        parts.append(sub.sample(take, seed=SEED))

In [52]:
sample = pl.concat(parts, how='diagonal_relaxed')
print('Sample:', sample.height, 'genes')

Sample: 1635 genes


In [53]:
def _amg_supporting_genes():
    frames = []
    for subset_dir in DRAMV_OUTPUT_DIR.iterdir():
        if not subset_dir.is_dir():
            continue
        for sample_dir in subset_dir.iterdir():
            f = sample_dir / 'distilled' / 'amg_summary.tsv'
            if not f.is_file():
                continue
            try:
                df = pl.read_csv(f, separator='\t')
            except Exception as e:
                print('skip', f, e)
                continue
            df = df.filter(pl.col('gene_id').str.starts_with('PF')) \
                   .select(pl.col('gene').alias('old_gene')).unique()
            df = df.with_columns(pl.lit(subset_dir.name).alias('subset'),
                                 pl.lit(sample_dir.name).alias('sample'))
            frames.append(df)
    return pl.concat(frames, how='diagonal_relaxed').unique()

amg_genes = _amg_supporting_genes()
amg_genes = amg_genes.join(
    mapping.select(['subset', 'sample', 'old_gene', 'new_gene']),
    on=['subset', 'sample', 'old_gene'], how='left').drop_nulls('new_gene')
print('AMG-supporting genes:', amg_genes.height)

AMG-supporting genes: 2482747


In [54]:
unified_genes = pl.concat([
    sample.select(['new_gene', 'subset', 'sample']).with_columns(pl.lit(True).alias('in_unified')),
    selected.select(['new_gene', 'subset', 'sample']).with_columns(pl.lit(True).alias('in_concordance')),
    amg_genes.select(['new_gene', 'subset', 'sample']).with_columns(pl.lit(True).alias('in_dramv_amg_calls')),
], how='diagonal_relaxed').unique(subset=['new_gene', 'subset', 'sample'], keep='first')

unified_genes = unified_genes.join(
    per_sample_genes_full.select(['new_gene', 'subset', 'sample', 'seq_type', 'environment', 'seq_len_aa']),
    on=['new_gene', 'subset', 'sample'], how='left',
)
unified_genes.write_parquet(UNIFIED_DIR.joinpath("unified_gene_set.parquet"))
print('unified gene set:', unified_genes.height)

UNIFIED_FA = UNIFIED_DIR.joinpath("unified_genes.fasta")
if not UNIFIED_FA.exists() or UNIFIED_FA.stat().st_size == 0:
    n = extract_subset_to_fasta(unified_genes, UNIFIED_FA)
    print('wrote', n, 'sequences ->', UNIFIED_FA)
else:
    print('reusing', UNIFIED_FA)

unified gene set: 2484029
reusing /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_pfam_analysis/unified_hmmsearch/unified_genes.fasta


In [55]:
def split_fasta(fa, n_chunks, out_dir):
    chunk_paths = [out_dir / f'chunk_{i:02d}.fasta' for i in range(n_chunks)]
    if all(p.exists() and p.stat().st_size > 0 for p in chunk_paths):
        return chunk_paths
    total = sum(1 for line in open(fa) if line.startswith('>'))
    per = (total + n_chunks - 1) // n_chunks
    handles = [open(p, 'w') for p in chunk_paths]
    count = 0
    cur_h = handles[0]
    with open(fa) as f:
        for line in f:
            if line.startswith('>'):
                cur_h = handles[min(count // per, n_chunks - 1)]
                count += 1
            cur_h.write(line)
    for h in handles:
        h.close()
    return chunk_paths

chunk_paths = split_fasta(UNIFIED_FA, N_CHUNKS, CHUNKS_DIR)
print(N_CHUNKS, 'chunks ready in', CHUNKS_DIR)

32 chunks ready in /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_pfam_analysis/unified_hmmsearch/chunks


In [56]:

def _hmmsearch_chunk(chunk, tbl, dom, cut_ga):
    if Path(tbl).exists() and Path(tbl).stat().st_size > 0:
        return tbl

    cmd = ['hmmsearch', '--cpu', str(INNER_CPUS), '--noali',
           '--tblout', str(tbl), '--domtblout', str(dom)]

    if cut_ga:
        cmd.append('--cut_ga')

    cmd.extend([str(PFAM_HMM), str(chunk)])

    with open(str(tbl) + '.stdout', 'w') as fh:
        subprocess.run(cmd, stdout=fh, check=True)

    return tbl

def parallel_hmmsearch(chunk_paths, merged_tbl, merged_dom, cut_ga, max_workers=None):
    if Path(merged_tbl).exists() and Path(merged_tbl).stat().st_size > 0:
        print('reusing', merged_tbl)
        return

    suffix = 'cutga' if cut_ga else 'default'
    per_t, per_d = [], []
    tasks = []

    for chunk in chunk_paths:
        t = chunk.parent / f'{chunk.stem}.{suffix}.tblout'
        d = chunk.parent / f'{chunk.stem}.{suffix}.domtblout'
        per_t.append(t)
        per_d.append(d)
        tasks.append((chunk, t, d, cut_ga))

    if max_workers is None:
        max_workers = min(len(tasks), (160/INNER_CPUS) or 1)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(_hmmsearch_chunk, *task)
            for task in tasks
        ]

        for f in as_completed(futures):
            print('  done', f.result())

    with open(merged_tbl, 'w') as out:
        for t in per_t:
            with open(t) as fh:
                for line in fh:
                    if not line.startswith('#'):
                        out.write(line)

    with open(merged_dom, 'w') as out:
        for d in per_d:
            with open(d) as fh:
                for line in fh:
                    if not line.startswith('#'):
                        out.write(line)

    print('merged ->', merged_tbl)

In [57]:
UNIFIED_DEFAULT_TBL = UNIFIED_DIR.joinpath("hmmsearch_default.tblout")
UNIFIED_CUTGA_TBL = UNIFIED_DIR.joinpath("hmmsearch_cutga.tblout")

In [58]:
parallel_hmmsearch(chunk_paths, UNIFIED_DEFAULT_TBL, UNIFIED_DIR.joinpath("hmmsearch_default.domtblout"), cut_ga=False)
parallel_hmmsearch(chunk_paths, UNIFIED_CUTGA_TBL, UNIFIED_DIR.joinpath("hmmsearch_cutga.domtblout"), cut_ga=True)

reusing /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_pfam_analysis/unified_hmmsearch/hmmsearch_default.tblout
reusing /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_pfam_analysis/unified_hmmsearch/hmmsearch_cutga.tblout


In [59]:
def parse_hmmsearch_tblout(path):
    """hmmsearch tbl: target=seq, query=HMM. f[0]=seq, f[3]=HMM acc, f[4]=E, f[5]=score."""
    rows = []
    with open(path) as fh:
        for line in fh:
            if line.startswith('#') or not line.strip():
                continue
            f = line.split()
            if len(f) < 9:
                continue
            rows.append({'query': f[0], 'tacc': f[3],
                         'evalue': float(f[4]), 'score': float(f[5])})
    if not rows:
        return pl.DataFrame(schema={'query': pl.Utf8, 'pfam_acc': pl.Utf8,
                                    'evalue': pl.Float64, 'score': pl.Float64})
    df = pl.from_dicts(rows).with_columns(
        pl.col('tacc').str.extract(r'(PF\d{5})', 1).alias('pfam_acc'))
    return df.group_by(['query', 'pfam_acc']).agg(
        pl.col('evalue').min(), pl.col('score').max())

In [60]:
unified_default_hits = parse_hmmsearch_tblout(UNIFIED_DEFAULT_TBL)
unified_cutga_hits = parse_hmmsearch_tblout(UNIFIED_CUTGA_TBL)

In [61]:
unified_default_hits.write_parquet(MAIN_DIR.joinpath("unified_hmmsearch_default_hits.parquet"))
unified_cutga_hits.write_parquet(MAIN_DIR.joinpath("unified_hmmsearch_cutga_hits.parquet"))
print('default:', unified_default_hits.height, 'cutga:', unified_cutga_hits.height)

default: 7221472 cutga: 1935418


In [62]:
def per_gene_counts(hits, name):
    return (
        hits.group_by('query').agg(
            pl.col('pfam_acc').unique().drop_nulls().alias(f'{name}_accs'))
        .with_columns(pl.col(f'{name}_accs').list.len().alias(f'{name}_n'))
        .rename({'query': 'new_gene'})
    )

unified_counts = (
    unified_genes
    .join(per_gene_counts(unified_default_hits, 'hmmscan_default'), on='new_gene', how='left')
    .join(per_gene_counts(unified_cutga_hits,   'hmmscan_cutga'),   on='new_gene', how='left')
    .with_columns(
        pl.col('hmmscan_default_n').fill_null(0).cast(pl.UInt32),
        pl.col('hmmscan_cutga_n').fill_null(0).cast(pl.UInt32),
        pl.col('hmmscan_default_accs').fill_null([]),
        pl.col('hmmscan_cutga_accs').fill_null([]),
    )
)

In [63]:
unified_counts.write_parquet(MAIN_DIR.joinpath("unified_hmmsearch_counts.parquet"))
print('per-gene counts:', unified_counts.height)

per-gene counts: 2484029


## Per-gene Pfam hit count distributions

Distributions of Pfam hit counts per gene, per tool, across the benchmark.

In [64]:
rows = []
for tool, col in [('DRAM-V', 'dramv_pfam_n'),
                  ('CheckAMG_all', 'checkamg_pfam_n_all'),
                  ('CheckAMG_keep', 'checkamg_pfam_n_keep')]:
    for st in ['metagenome', 'virome', 'viral_genome']:
        s = per_sample_genes_full.filter(pl.col('seq_type') == st)[col]
        rows.append({'tool': tool, 'seq_type': st, 'n': s.len(),
                     'mean':   float(s.mean() or 0),
                     'median': float(s.median() or 0),
                     'q95':    float(s.quantile(0.95) or 0),
                     'q99':    float(s.quantile(0.99) or 0),
                     'max':    int(s.max() or 0),
                     'frac_zero':  float((s == 0).sum() / max(s.len(), 1)),
                     'frac_gt_10': float((s > 10).sum() / max(s.len(), 1)),
                     'frac_gt_50': float((s > 50).sum() / max(s.len(), 1))})
for tool, col in [('hmmscan_default', 'hmmscan_default_n'),
                  ('hmmscan_cutga', 'hmmscan_cutga_n')]:
    for st in ['metagenome', 'virome', 'viral_genome']:
        s = unified_counts.filter(pl.col('seq_type') == st)[col]
        rows.append({'tool': tool, 'seq_type': st, 'n': s.len(),
                     'mean':   float(s.mean() or 0),
                     'median': float(s.median() or 0),
                     'q95':    float(s.quantile(0.95) or 0),
                     'q99':    float(s.quantile(0.99) or 0),
                     'max':    int(s.max() or 0),
                     'frac_zero':  float((s == 0).sum() / max(s.len(), 1)),
                     'frac_gt_10': float((s > 10).sum() / max(s.len(), 1)),
                     'frac_gt_50': float((s > 50).sum() / max(s.len(), 1))})

In [65]:
pfam_hits_summary = pl.DataFrame(rows)
pfam_hits_summary

tool,seq_type,n,mean,median,q95,q99,max,frac_zero,frac_gt_10,frac_gt_50
str,str,i64,f64,f64,f64,f64,i64,f64,f64,f64
"""DRAM-V""","""metagenome""",2128238,265.586123,290.0,295.0,297.0,300,0.000054,0.999247,0.987831
"""DRAM-V""","""virome""",2327154,269.915618,290.0,295.0,297.0,300,0.000053,0.999195,0.988488
"""DRAM-V""","""viral_genome""",200893,267.35396,289.0,295.0,297.0,300,0.00007,0.999632,0.994176
"""CheckAMG_all""","""metagenome""",2128238,0.867277,0.0,4.0,8.0,115,0.594212,0.00513,0.000033
"""CheckAMG_all""","""virome""",2327154,1.353347,1.0,5.0,10.0,116,0.422832,0.009001,0.000035
…,…,…,…,…,…,…,…,…,…,…
"""hmmscan_default""","""virome""",1100713,4.202664,2.0,16.0,32.0,404,0.184789,0.099205,0.002002
"""hmmscan_default""","""viral_genome""",169015,2.317593,1.0,8.0,20.0,219,0.294731,0.034387,0.001029
"""hmmscan_cutga""","""metagenome""",1214301,1.052768,0.0,4.0,8.0,58,0.507294,0.003517,0.000003


In [66]:
OUTPUT_PFAM_HITS_SUMMARY_TSV = OUTPUT_TABLES_DIR.joinpath("dramv_pfam_hits_summary_stats.tsv")

In [67]:
pfam_hits_summary.write_csv(OUTPUT_PFAM_HITS_SUMMARY_TSV, separator='\t')

In [68]:
# ECDF table thinned to 2K points per (tool, seq_type) for plotting
ecdf_parts = []
for tool, col, src in [('DRAM-V', 'dramv_pfam_n', per_sample_genes_full),
                       ('CheckAMG_all', 'checkamg_pfam_n_all', per_sample_genes_full),
                       ('CheckAMG_keep', 'checkamg_pfam_n_keep', per_sample_genes_full),
                       ('hmmscan_default', 'hmmscan_default_n', unified_counts),
                       ('hmmscan_cutga',   'hmmscan_cutga_n',   unified_counts)]:
    for st in ['metagenome', 'virome', 'viral_genome']:
        sub = src.filter(pl.col('seq_type') == st)
        if sub.height == 0:
            continue
        s = sub[col].sort()
        n = s.len()
        idx = list(range(0, n, max(1, n // 2000)))
        ecdf_parts.append(pl.DataFrame({
            'tool': [tool] * len(idx),
            'seq_type': [st] * len(idx),
            'value': [s[i] for i in idx],
            'ecdf':  [(i + 1) / n for i in idx],
        }))

In [69]:
ecdf = (
    pl.concat(ecdf_parts, how='diagonal_relaxed')
    .with_columns(
        pl.col("seq_type")
        .replace(seq_type_label)
        .cast(pl.Categorical)
    )
    .select([
        "tool",
        "seq_type",
        "value",
        "ecdf"
    ])
    .with_columns(
        pl.col("tool")
        .replace({
            "DRAM-V": "DRAM-V",
            "CheckAMG_all": "CheckAMG (all)",
            "CheckAMG_keep": "CheckAMG (kept)",
            "hmmscan_default": "HMMER (default)",
            "hmmscan_cutga": "HMMER (cut_ga)"
        })
        .cast(pl.Categorical)
    )
    .with_columns([
        pl.col("value").cast(pl.Int32),
        pl.col("ecdf").cast(pl.Float32)
    ])
)

In [70]:
ecdf

tool,seq_type,value,ecdf
cat,cat,i32,f32
"""DRAM-V""","""Mixed metagenomes""",0,4.6987e-7
"""DRAM-V""","""Mixed metagenomes""",8,0.0005
"""DRAM-V""","""Mixed metagenomes""",12,0.001
"""DRAM-V""","""Mixed metagenomes""",15,0.0015
"""DRAM-V""","""Mixed metagenomes""",18,0.002
…,…,…,…
"""HMMER (cut_ga)""","""Viral genomes""",5,0.997976
"""HMMER (cut_ga)""","""Viral genomes""",5,0.998474
"""HMMER (cut_ga)""","""Viral genomes""",6,0.998971


In [71]:
ecdf.write_csv(OUTPUT_TABLES_DIR.joinpath("dramv_pfam_ecdf_points.tsv"), separator='\t')

In [72]:
scatter = (
    unified_counts
    .join(
        per_sample_genes_full.select(['new_gene', 'dramv_pfam_n',
                                      'checkamg_pfam_n_all', 'checkamg_pfam_n_keep']),
        on='new_gene', how='left',
    )
    .select(['new_gene', 'seq_type', 'environment', 'dramv_pfam_n',
              'checkamg_pfam_n_all', 'checkamg_pfam_n_keep',
              'hmmscan_default_n', 'hmmscan_cutga_n'])
    .sort(['new_gene'])
)

In [73]:
scatter

new_gene,seq_type,environment,dramv_pfam_n,checkamg_pfam_n_all,checkamg_pfam_n_keep,hmmscan_default_n,hmmscan_cutga_n
str,str,str,u32,u32,u32,u32,u32
"""Ga0485157_0000001_10""","""metagenome""","""freshwater""",292,3,2,15,3
"""Ga0485157_0000001_100""","""metagenome""","""freshwater""",290,1,1,2,1
"""Ga0485157_0000001_101""","""metagenome""","""freshwater""",206,0,0,1,0
"""Ga0485157_0000001_102""","""metagenome""","""freshwater""",289,1,1,3,1
"""Ga0485157_0000001_103""","""metagenome""","""freshwater""",296,8,0,17,3
…,…,…,…,…,…,…,…
"""scaffold_9_c1_99""","""metagenome""","""soil""",292,0,0,3,1
"""scaffold_9_c1_99""","""metagenome""","""soil""",293,1,0,3,1
"""scaffold_9_c1_99""","""metagenome""","""soil""",290,2,0,3,1


In [74]:
scatter.write_parquet(OUTPUT_TABLES_DIR.joinpath("dramv_pfam_scatter.parquet"))

## Distinct Pfam clans per gene

Distinct clans per gene, after mapping each Pfam accession to its clan.

In [75]:
PFAM_CLANS = Path('./data/Pfam-A.clans.tsv')

In [76]:
clan_df = pl.read_csv(
    PFAM_CLANS, separator='\t', has_header=False,
    new_columns=['pfam_acc', 'clan_acc', 'clan_id', 'pfam_id', 'pfam_description'],
    null_values=['', '\\N'],
).with_columns(
    pl.when(pl.col('clan_acc').is_null())
    .then(pl.col('pfam_acc'))
    .otherwise(pl.col('clan_acc')).alias('effective_clan')
)

In [77]:
pfam_to_clan = dict(zip(clan_df['pfam_acc'].to_list(),
                        clan_df['effective_clan'].to_list()))
print(f'{clan_df.height} pfams mapped to {clan_df["effective_clan"].n_unique()} effective clans')

30134 pfams mapped to 17092 effective clans


In [78]:
SAMPLE_PER_SEQTYPE = 100_000

In [79]:
def acc_list_to_clan_list(accs):
    if accs is None:
        return []
    seen = set()
    for a in accs:
        c = pfam_to_clan.get(a)
        if c is not None:
            seen.add(c)
    return sorted(seen)

clan_parts = []
for st in ['metagenome', 'virome', 'viral_genome']:
    sub = per_sample_genes_full.filter(pl.col('seq_type') == st)
    take = min(SAMPLE_PER_SEQTYPE, sub.height)
    if take:
        clan_parts.append(sub.sample(take, seed=SEED))
clan_sample = pl.concat(clan_parts, how='diagonal_relaxed')

clans = (
    clan_sample.with_columns(
        pl.col('dramv_pfam_accessions').map_elements(acc_list_to_clan_list,
                                                     return_dtype=pl.List(pl.Utf8)).alias('dramv_clans'),
        pl.col('checkamg_pfam_accessions_all').map_elements(acc_list_to_clan_list,
                                                            return_dtype=pl.List(pl.Utf8)).alias('checkamg_all_clans'),
        pl.col('checkamg_pfam_accessions_keep').map_elements(acc_list_to_clan_list,
                                                             return_dtype=pl.List(pl.Utf8)).alias('checkamg_keep_clans'),
    ).with_columns(
        pl.col('dramv_clans').list.len().alias('dramv_clan_n'),
        pl.col('checkamg_all_clans').list.len().alias('checkamg_all_clan_n'),
        pl.col('checkamg_keep_clans').list.len().alias('checkamg_keep_clan_n'),
    ).select([
        'new_gene', 'seq_type', 'environment', 'seq_len_aa',
        'dramv_pfam_n', 'checkamg_pfam_n_all', 'checkamg_pfam_n_keep',
        'dramv_clan_n', 'checkamg_all_clan_n', 'checkamg_keep_clan_n',
    ])
)

In [80]:
clans

new_gene,seq_type,environment,seq_len_aa,dramv_pfam_n,checkamg_pfam_n_all,checkamg_pfam_n_keep,dramv_clan_n,checkamg_all_clan_n,checkamg_keep_clan_n
str,str,str,i64,u32,u32,u32,u64,u64,u64
"""Ga0485158_0002497_7""","""metagenome""","""freshwater""",62,213,0,0,173,0,0
"""Ga0485159_0001044_11""","""metagenome""","""freshwater""",64,181,0,0,155,0,0
"""Ga0485159_0000254_19""","""metagenome""","""freshwater""",326,291,1,1,231,1,1
"""scaffold_29212_c1_4""","""metagenome""","""soil""",299,289,2,1,224,1,1
"""scaffold_11_c2_7""","""metagenome""","""soil""",133,295,0,0,245,0,0
…,…,…,…,…,…,…,…,…,…
"""IMGVR_UViG_3300039146_000001_3300039146_Ga0237833_00001_1""","""viral_genome""","""marine""",181,298,2,1,242,1,1
"""IMGVR_UViG_3300035562_000121_3300035562_Ga0376655_0000038_5253-72841_65""","""viral_genome""","""soil""",329,293,1,1,223,1,1
"""IMGVR_UViG_3300045988_147200_3300045988_Ga0495776_122165_44""","""viral_genome""","""gut""",228,288,1,0,240,1,0


In [81]:
clans.write_csv(OUTPUT_TABLES_DIR.joinpath("dramv_pfam_clans_per_gene.tsv"), separator="\t")

In [82]:
clans.group_by('seq_type').agg(
    pl.col('dramv_clan_n').mean().alias('dramv_clan_mean'),
    pl.col('dramv_clan_n').quantile(0.95).alias('dramv_clan_q95'),
    (pl.col('dramv_clan_n') >= 10).sum().alias('dramv_ge10_clans'),
    pl.col('checkamg_keep_clan_n').mean().alias('checkamg_keep_clan_mean'),
    pl.col('checkamg_keep_clan_n').max().alias('checkamg_keep_clan_max'),
).sort('seq_type')

seq_type,dramv_clan_mean,dramv_clan_q95,dramv_ge10_clans,checkamg_keep_clan_mean,checkamg_keep_clan_max
str,f64,f64,u64,f64,u64
"""metagenome""",211.06233,245.0,99929,0.19689,2
"""viral_genome""",213.76201,245.0,99966,0.15879,2
"""virome""",212.98872,244.0,99920,0.32919,3


In [83]:
rows = []

for tool, col, df in [
    ('DRAM-V', 'dramv_pfam_n', per_sample_genes_full),
    ('CheckAMG_all', 'checkamg_pfam_n_all', per_sample_genes_full),
    ('CheckAMG_keep', 'checkamg_pfam_n_keep', per_sample_genes_full),
    ('hmmscan_default', 'hmmscan_default_n', unified_counts),
    ('hmmscan_cutga', 'hmmscan_cutga_n', unified_counts),
]:
    for st in ['metagenome', 'virome', 'viral_genome']:
        s = df.filter(pl.col('seq_type') == st)[col]
        rows.append({
            'tool': tool,
            'metric': 'pfam',
            'seq_type': st,
            'n': s.len(),
            'mean': float(s.mean() or 0),
            'median': float(s.median() or 0),
            'q95': float(s.quantile(0.95) or 0),
            'q99': float(s.quantile(0.99) or 0),
            'max': int(s.max() or 0),
            'frac_zero': float((s == 0).sum() / max(s.len(), 1)),
            'frac_gt_10': float((s > 10).sum() / max(s.len(), 1)),
            'frac_gt_50': float((s > 50).sum() / max(s.len(), 1)),
        })

# Clan summaries (only tools that have clans)
for tool, col in [
    ('DRAM-V', 'dramv_clan_n'),
    ('CheckAMG_all', 'checkamg_all_clan_n'),
    ('CheckAMG_keep', 'checkamg_keep_clan_n'),
]:
    for st in ['metagenome', 'virome', 'viral_genome']:
        s = clans.filter(pl.col('seq_type') == st)[col]
        rows.append({
            'tool': tool,
            'metric': 'clan',
            'seq_type': st,
            'n': s.len(),
            'mean': float(s.mean() or 0),
            'median': float(s.median() or 0),
            'q95': float(s.quantile(0.95) or 0),
            'q99': float(s.quantile(0.99) or 0),
            'max': int(s.max() or 0),
            'frac_zero': float((s == 0).sum() / max(s.len(), 1)),
            'frac_gt_10': float((s > 10).sum() / max(s.len(), 1)),
            'frac_gt_50': float((s > 50).sum() / max(s.len(), 1)),
        })

pfam_hits_summary_with_clans = (
    pl.DataFrame(rows)
    .sort(["tool", "metric", "seq_type"])
)
pfam_hits_summary_with_clans

tool,metric,seq_type,n,mean,median,q95,q99,max,frac_zero,frac_gt_10,frac_gt_50
str,str,str,i64,f64,f64,f64,f64,i64,f64,f64,f64
"""CheckAMG_all""","""clan""","""metagenome""",100000,0.5292,0.0,2.0,3.0,35,0.59313,0.00005,0.0
"""CheckAMG_all""","""clan""","""viral_genome""",100000,0.55845,0.0,2.0,3.0,65,0.56168,0.00039,0.00001
"""CheckAMG_all""","""clan""","""virome""",100000,0.78009,1.0,2.0,4.0,40,0.4239,0.00011,0.0
"""CheckAMG_all""","""pfam""","""metagenome""",2128238,0.867277,0.0,4.0,8.0,115,0.594212,0.00513,0.000033
"""CheckAMG_all""","""pfam""","""viral_genome""",200893,0.769071,0.0,3.0,6.0,68,0.56114,0.002116,0.000005
…,…,…,…,…,…,…,…,…,…,…,…
"""hmmscan_cutga""","""pfam""","""viral_genome""",169015,0.406579,0.0,2.0,3.0,30,0.691223,0.000106,0.0
"""hmmscan_cutga""","""pfam""","""virome""",1100713,1.206434,1.0,5.0,8.0,55,0.441333,0.004287,0.000003
"""hmmscan_default""","""pfam""","""metagenome""",1214301,4.376749,2.0,16.0,31.0,360,0.178162,0.101864,0.001972


In [84]:
pfam_hits_summary_with_clans.write_csv(OUTPUT_TABLES_DIR.joinpath("dramv_pfam_hits_clans_summary.tsv"), separator="\t")

## HMMER support for DRAM-V Pfam-based AMG calls

For every (gene, Pfam accession) row in DRAM-V's `distilled/amg_summary.tsv`, classify whether stock `hmmsearch` recovers the same Pfam hit on the same sequence: supported (passes `--cut_ga`), borderline (default thresholds only), or rejected (no hit at all).

In [85]:
def load_dramv_amg_calls():
    frames = []
    for subset_dir in DRAMV_OUTPUT_DIR.iterdir():
        if not subset_dir.is_dir():
            continue
        for sample_dir in subset_dir.iterdir():
            f = sample_dir / 'distilled' / 'amg_summary.tsv'
            if not f.is_file():
                continue
            try:
                df = pl.read_csv(f, separator='\t',
                                 schema_overrides={'auxiliary_score': pl.Float64})
            except Exception as e:
                print('skip', f, e)
                continue
            df = df.filter(pl.col('gene_id').str.starts_with('PF'))
            df = df.with_columns(pl.lit(subset_dir.name).alias('subset'),
                                 pl.lit(sample_dir.name).alias('sample')) \
                   .rename({'gene': 'old_gene', 'gene_id': 'pfam_acc'})
            df = df.with_columns(pl.col('pfam_acc').str.extract(r'(PF\d{5})', 1))
            frames.append(df)
    return pl.concat(frames, how='diagonal_relaxed')

In [86]:
amg_calls = load_dramv_amg_calls()
amg_calls = amg_calls.join(mapping.select(['subset', 'sample', 'old_gene', 'new_gene']),
                           on=['subset', 'sample', 'old_gene'], how='left')
print('Pfam-based AMG calls:', amg_calls.height)

Pfam-based AMG calls: 9974363


In [87]:
amg_calls = amg_calls.join(
    unified_default_hits.rename({'query': 'new_gene', 'evalue': 'hmmer_evalue', 'score': 'hmmer_score'})
                       .select(['new_gene', 'pfam_acc', 'hmmer_evalue', 'hmmer_score']),
    on=['new_gene', 'pfam_acc'], how='left',
)
amg_calls = amg_calls.join(
    unified_cutga_hits.select(['query', 'pfam_acc']).rename({'query': 'new_gene'})
                     .with_columns(pl.lit(True).alias('passes_cut_ga')),
    on=['new_gene', 'pfam_acc'], how='left',
).with_columns(pl.col('passes_cut_ga').fill_null(False))

amg_calls = amg_calls.with_columns(
    pl.col('subset').map_elements(seq_type_of, return_dtype=pl.Utf8).alias('seq_type'),
    pl.col('sample').map_elements(environment_of, return_dtype=pl.Utf8).alias('environment'),
    pl.when(pl.col('hmmer_evalue').is_null()).then(pl.lit('hmmer_rejected'))
    .when(pl.col('passes_cut_ga')).then(pl.lit('hmmer_supported'))
    .otherwise(pl.lit('hmmer_borderline')).alias('support_class'),
)
print('total Pfam-based DRAM-V AMG calls:', amg_calls.height)

total Pfam-based DRAM-V AMG calls: 9974363


In [88]:
amg_calls

old_gene,pfam_acc,scaffold,auxiliary_score,amg_flags,gene_description,module,sheet,header,subheader,potential_amg,gene_id_origin,metabolism,reference,verified,subset,sample,new_gene,hmmer_evalue,hmmer_score,passes_cut_ga,seq_type,environment,support_class
str,str,str,f64,str,str,str,str,str,str,bool,str,str,str,str,str,str,str,f64,f64,bool,str,str,str
"""scaffold_10000_c1-cat_2_2""","""PF01077""","""scaffold_10000_c1-cat_2""",4.0,"""MKTFB""","""rdsrA; reverse-acting dissimilatory sulfite reductase (alpha subunit)""",null,null,null,null,null,"""amg_database""",null,"""Anantharaman et al. 2014""","""False""","""metagenomes""","""soil_T42_15_3_57""","""scaffold_10000_c1_2""",null,null,false,"""metagenome""","""soil""","""hmmer_rejected"""
"""scaffold_10000_c1-cat_2_2""","""PF01717""","""scaffold_10000_c1-cat_2""",4.0,"""MKTFB""","""Cobalamin-independent synthase, Catalytic domain""",null,null,null,null,null,"""amg_database""",null,"""Roux et al. 2016""","""False""","""metagenomes""","""soil_T42_15_3_57""","""scaffold_10000_c1_2""",null,null,false,"""metagenome""","""soil""","""hmmer_rejected"""
"""scaffold_10000_c1-cat_2_2""","""PF03332""","""scaffold_10000_c1-cat_2""",4.0,"""MKTFB""","""Eukaryotic phosphomannomutase""",null,null,null,null,null,"""amg_database""","""M""","""Roux et al. 2016""","""False""","""metagenomes""","""soil_T42_15_3_57""","""scaffold_10000_c1_2""",null,null,false,"""metagenome""","""soil""","""hmmer_rejected"""
"""scaffold_10000_c1-cat_2_2""","""PF01259""","""scaffold_10000_c1-cat_2""",4.0,"""MKTFB""","""SAICAR synthetase""",null,null,null,null,null,"""amg_database""",null,"""Roux et al. 2016""","""False""","""metagenomes""","""soil_T42_15_3_57""","""scaffold_10000_c1_2""",null,null,false,"""metagenome""","""soil""","""hmmer_rejected"""
"""scaffold_10000_c1-cat_2_2""","""PF13501""","""scaffold_10000_c1-cat_2""",4.0,"""MKTFB""","""Sulfur oxidation protein SoxY""",null,null,null,null,null,"""amg_database""","""M""","""Roux et al. 2016""","""False""","""metagenomes""","""soil_T42_15_3_57""","""scaffold_10000_c1_2""",null,null,false,"""metagenome""","""soil""","""hmmer_rejected"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SRR9161509_k127_99934-cat_3_93""","""PF04131""","""SRR9161509_k127_99934-cat_3""",4.0,"""MKTFB""","""Putative N-acetylmannosamine-6-phosphate epimerase""",null,null,null,null,null,"""amg_database""",null,"""Roux et al. 2016""","""False""","""viromes""","""human_gut_SRR9161509_contigs""","""SRR9161509_k127_99934_93""",null,null,false,"""virome""","""gut""","""hmmer_rejected"""
"""SRR9161509_k127_99934-cat_3_94""","""PF13091""","""SRR9161509_k127_99934-cat_3""",4.0,"""MKTFB""","""PLD-like domain""",null,null,null,null,null,"""amg_database""",null,"""Roux et al. 2016""","""False""","""viromes""","""human_gut_SRR9161509_contigs""","""SRR9161509_k127_99934_94""",null,null,false,"""virome""","""gut""","""hmmer_rejected"""
"""SRR9161509_k127_99934-cat_3_94""","""PF02401""","""SRR9161509_k127_99934-cat_3""",4.0,"""MKTFB""","""LytB protein""",null,null,null,null,null,"""amg_database""","""M""","""Roux et al. 2016""","""False""","""viromes""","""human_gut_SRR9161509_contigs""","""SRR9161509_k127_99934_94""",null,null,false,"""virome""","""gut""","""hmmer_rejected"""


In [89]:
amg_calls.write_parquet(OUTPUT_TABLES_DIR.joinpath("dramv_pfam_amg_calls.parquet"))

In [90]:
amg_calls.group_by(['seq_type', 'support_class']).len().sort(['seq_type', 'support_class'])

seq_type,support_class,len
str,str,u64
"""metagenome""","""hmmer_borderline""",23614
"""metagenome""","""hmmer_rejected""",4794437
"""metagenome""","""hmmer_supported""",49913
"""viral_genome""","""hmmer_borderline""",1094
"""viral_genome""","""hmmer_rejected""",621630
"""viral_genome""","""hmmer_supported""",975
"""virome""","""hmmer_borderline""",40842
"""virome""","""hmmer_rejected""",4348300
"""virome""","""hmmer_supported""",93558


In [91]:
support_summary = amg_calls.group_by('support_class').len().with_columns(
    (pl.col('len') / amg_calls.height).alias('frac')
).sort('support_class')
support_summary

support_class,len,frac
str,u64,f64
"""hmmer_borderline""",65550,0.006572
"""hmmer_rejected""",9764367,0.978946
"""hmmer_supported""",144446,0.014482


In [92]:
SUPPORT_SUMMARY_TSV = OUTPUT_TABLES_DIR.joinpath("dramv_pfam_support_summary.tsv")

In [93]:
support_summary.write_csv(SUPPORT_SUMMARY_TSV, separator='\t')

In [94]:
top_pfam = (
    amg_calls.group_by(['pfam_acc', 'support_class']).len()
    .pivot(values='len', index='pfam_acc', on='support_class').fill_null(0)
)
support_cols = [c for c in ('hmmer_supported', 'hmmer_borderline', 'hmmer_rejected')
                if c in top_pfam.columns]
top_pfam = top_pfam.with_columns(sum(pl.col(c) for c in support_cols).alias('total'))
if 'hmmer_rejected' in top_pfam.columns:
    top_pfam = top_pfam.with_columns(
        (pl.col('hmmer_rejected') / pl.col('total')).alias('frac_rejected'))
top_pfam = top_pfam.sort('total', descending=True)

In [95]:
top_pfam

pfam_acc,hmmer_rejected,hmmer_supported,hmmer_borderline,total,frac_rejected
str,u64,u64,u64,u64,f64
"""PF03659""",128006,9,30,128045,0.999695
"""PF01474""",108794,16,12,108822,0.999743
"""PF01384""",102503,190,51,102744,0.997654
"""PF02224""",94983,240,2193,97416,0.975025
"""PF07971""",95175,207,16,95398,0.997662
…,…,…,…,…,…
"""PF13417""",7450,689,155,8294,0.89824
"""PF09489""",7142,9,38,7189,0.993462
"""PF00132""",2191,3098,128,5417,0.404467


In [96]:
TOP_PFAM_TSV = OUTPUT_TABLES_DIR.joinpath("dramv_pfam_support_by_pfam.tsv")

In [97]:
top_pfam.write_csv(TOP_PFAM_TSV, separator='\t')

## Sequence-level mechanism

Gene length and amino-acid Shannon entropy on a stratified subsample of the unified gene set, correlated with DRAM-V's hit count and with its excess over stock HMMER.

In [98]:
import math
from collections import Counter

def aa_entropy(seq):
    if not seq:
        return 0.0
    c = Counter(seq)
    n = sum(c.values())
    return -sum((v / n) * math.log2(v / n) for v in c.values())

In [99]:
import random
random.seed(20260426)
NPB = 2000

In [100]:
lookup = dict(zip(per_sample_genes_full['new_gene'].to_list(),
                  per_sample_genes_full['dramv_pfam_n'].to_list()))

def bucket(n):
    if n is None:
        return None
    if n == 0:
        return '0'
    if n <= 5:
        return '1-5'
    if n <= 20:
        return '6-20'
    if n <= 100:
        return '21-100'
    return '100+'

names_by_bucket = {}
with open(UNIFIED_FA) as fh:
    for line in fh:
        if line.startswith('>'):
            name = line[1:].split()[0]
            b = bucket(lookup.get(name))
            if b:
                names_by_bucket.setdefault(b, []).append(name)
selected = set()
for b, names in names_by_bucket.items():
    random.shuffle(names)
    selected.update(names[:NPB])
print('Selected genes:', len(selected))

Selected genes: 5493


In [101]:
SUBSAMPLE_FA = MAIN_DIR.joinpath("sequence_features", "subsample.fasta")
SUBSAMPLE_FA.parent.mkdir(parents=True, exist_ok=True)

In [102]:
n_written = 0
if not SUBSAMPLE_FA.exists() or SUBSAMPLE_FA.stat().st_size == 0:
    with open(UNIFIED_FA) as fh, open(SUBSAMPLE_FA, 'w') as out:
        cur, lines = None, []
        for line in fh:
            if line.startswith('>'):
                if cur in selected:
                    out.write(f'>{cur}\n' + ''.join(lines))
                    n_written += 1
                cur = line[1:].split()[0]
                lines = []
            else:
                lines.append(line)
        if cur in selected:
            out.write(f'>{cur}\n' + ''.join(lines))
            n_written += 1
print('wrote', SUBSAMPLE_FA, n_written, 'sequences')

wrote /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_pfam_analysis/sequence_features/subsample.fasta 0 sequences


In [103]:
rows = []
with open(SUBSAMPLE_FA) as fh:
    cur, lines = None, []
    for line in fh:
        if line.startswith('>'):
            if cur is not None:
                seq = ''.join(lines).rstrip('*').replace('\n', '')
                rows.append({'new_gene': cur, 'length': len(seq),
                             'aa_entropy': aa_entropy(seq)})
            cur = line[1:].split()[0]
            lines = []
        else:
            lines.append(line.strip())
    if cur is not None:
        seq = ''.join(lines).rstrip('*').replace('\n', '')
        rows.append({'new_gene': cur, 'length': len(seq),
                     'aa_entropy': aa_entropy(seq)})
features = pl.DataFrame(rows)

sequence_features = (
    features
    .join(per_sample_genes_full.select(['new_gene', 'seq_type', 'environment',
                                        'dramv_pfam_n', 'checkamg_pfam_n_all',
                                        'checkamg_pfam_n_keep']),
          on='new_gene', how='left')
    .join(unified_counts.select(['new_gene', 'hmmscan_default_n', 'hmmscan_cutga_n']),
          on='new_gene', how='left')
    .with_columns((pl.col('dramv_pfam_n') - pl.col('hmmscan_default_n'))
                  .alias('excess_dramv_vs_hmmer'))
)

In [104]:
sequence_features

new_gene,length,aa_entropy,seq_type,environment,dramv_pfam_n,checkamg_pfam_n_all,checkamg_pfam_n_keep,hmmscan_default_n,hmmscan_cutga_n,excess_dramv_vs_hmmer
str,i64,f64,str,str,u32,u32,u32,u32,u32,u32
"""scaffold_1_c1_186""",163,3.959661,"""metagenome""","""soil""",292,0,0,5,0,287
"""scaffold_1_c1_186""",163,3.959661,"""metagenome""","""soil""",292,0,0,5,0,287
"""scaffold_1_c1_186""",163,3.959661,"""metagenome""","""soil""",292,0,0,5,0,287
"""scaffold_1_c1_186""",163,3.959661,"""metagenome""","""soil""",294,2,0,5,0,289
"""scaffold_1_c1_186""",163,3.959661,"""metagenome""","""soil""",294,2,0,5,0,289
…,…,…,…,…,…,…,…,…,…,…
"""scaffold_9775_c1_3""",171,4.084986,"""virome""","""soil""",292,0,0,11,5,281
"""scaffold_9775_c1_3""",171,4.084986,"""virome""","""soil""",293,3,1,11,5,282
"""scaffold_9775_c1_3""",171,4.084986,"""virome""","""soil""",293,3,1,11,5,282


In [105]:
SEQUENCE_FEATURES_PARQUET = OUTPUT_TABLES_DIR.joinpath("dramv_pfam_per_gene_features.parquet")

In [106]:
sequence_features.write_parquet(SEQUENCE_FEATURES_PARQUET)

In [107]:
import numpy as np
from scipy.stats import spearmanr

def show_rho(y_col, x_col):
    sub = sequence_features.drop_nulls([x_col, y_col])
    if sub.height < 10:
        return
    rho, p = spearmanr(sub[x_col].cast(pl.Float64).to_numpy(),
                       sub[y_col].cast(pl.Float64).to_numpy())
    print(f'  {y_col:25s} ~ {x_col:18s}  rho={rho:+.3f}  p={p:.2e}  n={sub.height}')

print('Spearman correlations:')
for y in ['dramv_pfam_n', 'excess_dramv_vs_hmmer']:
    for x in ['length', 'aa_entropy']:
        show_rho(y, x)

Spearman correlations:
  dramv_pfam_n              ~ length              rho=+0.199  p=0.00e+00  n=66094
  dramv_pfam_n              ~ aa_entropy          rho=+0.145  p=1.07e-305  n=66094
  excess_dramv_vs_hmmer     ~ length              rho=+0.120  p=2.25e-211  n=66094
  excess_dramv_vs_hmmer     ~ aa_entropy          rho=+0.111  p=3.74e-180  n=66094


## Score calibration on concordant hits

For (gene, Pfam) pairs that BOTH DRAM-V and stock HMMER recover, compare HMMER bitscores (log-odds) to DRAM-V's MMseqs2 profile bitscores (sum-of-substitution-scores).

In [108]:
supported = amg_calls.filter(pl.col('support_class').is_in(
    ['hmmer_supported', 'hmmer_borderline']))
key_set = set(zip(supported['subset'].to_list(), supported['sample'].to_list(),
                  supported['old_gene'].to_list(), supported['pfam_acc'].to_list()))
print('supported/borderline AMG rows:', supported.height,
      'with', len(set((s, sa) for s, sa, _, _ in key_set)), 'samples')

supported/borderline AMG rows: 209996 with 21 samples


In [109]:
b6_frames = []
for subset_dir in DRAMV_RAW_ANNOTS_DIR.iterdir():
    if not subset_dir.is_dir():
        continue
    for sample_dir in subset_dir.iterdir():
        b6 = sample_dir / 'pfam_output.b6'
        if not b6.is_file():
            continue
        sub_keys = {(g, p) for s, sa, g, p in key_set
                    if s == subset_dir.name and sa == sample_dir.name}
        if not sub_keys:
            continue
        rows = []
        with open(b6) as fh:
            for line in fh:
                tab = line.split('\t')
                if len(tab) < 12:
                    continue
                m = re.search(r'PF\d{5}', tab[1])
                if not m:
                    continue
                p = m.group(0)
                if (tab[0], p) in sub_keys:
                    rows.append({'old_gene': tab[0], 'pfam_acc': p,
                                 'mmseqs_evalue': float(tab[10]),
                                 'mmseqs_bits':   float(tab[11])})
        if rows:
            df = pl.from_dicts(rows).with_columns(
                pl.lit(subset_dir.name).alias('subset'),
                pl.lit(sample_dir.name).alias('sample'),
            ).group_by(['subset', 'sample', 'old_gene', 'pfam_acc']).agg(
                pl.col('mmseqs_evalue').min(), pl.col('mmseqs_bits').max(),
            )
            print(f'  {subset_dir.name}/{sample_dir.name}: matched {df.height}')
            b6_frames.append(df)

  metagenomes/soil_T42_15_3_57: matched 6167


  metagenomes/soil_T42_15_2_56: matched 8219


  metagenomes/soil_T42_15_1_55: matched 3439


  metagenomes/human_gut_SRR9162908_contigs: matched 1601


  metagenomes/freshwater_Ga0485157_contigs: matched 19307


...

  viromes/soil_T42_15_3_45: matched 2573


  viromes/soil_T42_15_2_44: matched 7786


  viromes/freshwater_Ga0485175_contigs: matched 25424


  viromes/human_gut_SRR9161506_contigs: matched 22021


  viromes/human_gut_SRR9161509_contigs: matched 14183


In [110]:
b6_joined = pl.concat(b6_frames, how='diagonal_relaxed') if b6_frames else pl.DataFrame(
    schema={'subset': pl.Utf8, 'sample': pl.Utf8, 'old_gene': pl.Utf8,
            'pfam_acc': pl.Utf8, 'mmseqs_evalue': pl.Float64, 'mmseqs_bits': pl.Float64})

In [111]:
calibrated_scores = supported.join(
    b6_joined, on=['subset', 'sample', 'old_gene', 'pfam_acc'], how='left'
).select(['subset', 'sample', 'seq_type', 'environment', 'old_gene', 'new_gene',
          'pfam_acc', 'support_class', 'hmmer_evalue', 'hmmer_score',
          'mmseqs_evalue', 'mmseqs_bits'])


In [112]:
calibrated_scores

subset,sample,seq_type,environment,old_gene,new_gene,pfam_acc,support_class,hmmer_evalue,hmmer_score,mmseqs_evalue,mmseqs_bits
str,str,str,str,str,str,str,str,f64,f64,f64,f64
"""metagenomes""","""soil_T42_15_3_57""","""metagenome""","""soil""","""scaffold_10019_c1-cat_2_9""","""scaffold_10019_c1_9""","""PF01242""","""hmmer_supported""",8.9000e-17,63.2,0.0,1129.0
"""metagenomes""","""soil_T42_15_3_57""","""metagenome""","""soil""","""scaffold_10019_c1-cat_2_10""","""scaffold_10019_c1_10""","""PF06508""","""hmmer_supported""",2.0000e-51,176.5,0.0,1607.0
"""metagenomes""","""soil_T42_15_3_57""","""metagenome""","""soil""","""scaffold_10019_c1-cat_2_10""","""scaffold_10019_c1_10""","""PF00733""","""hmmer_supported""",0.000048,25.3,0.0,1963.0
"""metagenomes""","""soil_T42_15_3_57""","""metagenome""","""soil""","""scaffold_10019_c1-cat_2_11""","""scaffold_10019_c1_11""","""PF01227""","""hmmer_supported""",2.2000e-68,231.3,0.0,1049.0
"""metagenomes""","""soil_T42_15_3_57""","""metagenome""","""soil""","""scaffold_1002_c1-cat_1_21""","""scaffold_1002_c1_21""","""PF00534""","""hmmer_supported""",6.4000e-20,73.4,0.0,2279.0
…,…,…,…,…,…,…,…,…,…,…,…
"""viromes""","""human_gut_SRR9161509_contigs""","""virome""","""gut""","""SRR9161509_k127_99934-cat_3_17""","""SRR9161509_k127_99934_17""","""PF00106""","""hmmer_supported""",9.8000e-53,180.6,0.0,1776.0
"""viromes""","""human_gut_SRR9161509_contigs""","""virome""","""gut""","""SRR9161509_k127_99934-cat_3_17""","""SRR9161509_k127_99934_17""","""PF02826""","""hmmer_borderline""",0.45,11.9,0.0,1369.0
"""viromes""","""human_gut_SRR9161509_contigs""","""virome""","""gut""","""SRR9161509_k127_99934-cat_3_21""","""SRR9161509_k127_99934_21""","""PF09290""","""hmmer_borderline""",0.53,12.6,3.1240e-235,707.0


In [113]:
calibrated_scores.write_parquet(OUTPUT_TABLES_DIR.joinpath("dramv_pfam_score_comparison.parquet"))

In [114]:
paired = calibrated_scores.drop_nulls(['hmmer_score', 'mmseqs_bits'])
if paired.height:
    rho, p = spearmanr(paired['hmmer_score'].cast(pl.Float64).to_numpy(),
                       paired['mmseqs_bits'].cast(pl.Float64).to_numpy())
    print(f'Spearman(hmmer_score, mmseqs_bits)  rho={rho:.3f}  p={p:.2e}  n={paired.height}')
    paired.select([
        pl.col('hmmer_score').min().alias('hmmer_score_min'),
        pl.col('hmmer_score').median().alias('hmmer_score_median'),
        pl.col('hmmer_score').max().alias('hmmer_score_max'),
        pl.col('mmseqs_bits').min().alias('mmseqs_bits_min'),
        pl.col('mmseqs_bits').median().alias('mmseqs_bits_median'),
        pl.col('mmseqs_bits').max().alias('mmseqs_bits_max'),
        (pl.col('mmseqs_evalue') == 0).sum().alias('mmseqs_evalue_zero_count'),
        pl.len().alias('n'),
    ])

Spearman(hmmer_score, mmseqs_bits)  rho=0.116  p=0.00e+00  n=209996


## Summary numbers

In [115]:
dramv = per_sample_genes_full['dramv_pfam_n']
checkamg_all = per_sample_genes_full['checkamg_pfam_n_all']
checkamg_keep = per_sample_genes_full['checkamg_pfam_n_keep']
hmmer = unified_counts['hmmscan_default_n']
hmmer_cutga = unified_counts['hmmscan_cutga_n']
amg = amg_calls

print('--- Benchmark-wide per-gene Pfam hit counts ---')
print(f'  DRAM-V           n={dramv.len():,}  mean={float(dramv.mean()):.2f}'
      f'  median={float(dramv.median()):.0f}  q95={float(dramv.quantile(0.95)):.0f}  max={int(dramv.max())}')
print(f'  CheckAMG (all)   n={checkamg_all.len():,}  mean={float(checkamg_all.mean()):.2f}  '
      f'  median={float(checkamg_all.median()):.0f}    q95={float(checkamg_all.quantile(0.95)):.0f}    max={int(checkamg_all.max())}')
print(f'  CheckAMG (kept)  n={checkamg_keep.len():,}  mean={float(checkamg_keep.mean()):.2f}  '
      f'  median={float(checkamg_keep.median()):.0f}    q95={float(checkamg_keep.quantile(0.95)):.0f}    max={int(checkamg_keep.max())}')
print(f'  hmmscan          n={hmmer.len():,}  mean={float(hmmer.mean()):.2f}  '
      f'  median={float(hmmer.median()):.0f}    q95={float(hmmer.quantile(0.95)):.0f}   max={int(hmmer.max())}')
print(f'  hmmscan (cut_ga) n={hmmer_cutga.len():,}  mean={float(hmmer_cutga.mean()):.2f}  '
      f'  median={float(hmmer_cutga.median()):.0f}    q95={float(hmmer_cutga.quantile(0.95)):.0f}    max={int(hmmer_cutga.max())}')

print('\n--- Distinct clans per gene (DRAM-V) ---')
dc = clans['dramv_clan_n']
print(f'  n={dc.len():,}  mean={float(dc.mean()):.2f}  median={float(dc.median()):.0f}  q95={float(dc.quantile(0.95)):.0f}  max={int(dc.max())}')
print(f'  fraction of genes with >= 10 clans: {(dc >= 10).sum() / dc.len() * 100:.2f}%')

print('\n--- Distinct clans per gene (CheckAMG, all) ---')
dc = clans['checkamg_all_clan_n']
print(f'  n={dc.len():,}  mean={float(dc.mean()):.2f}  median={float(dc.median()):.0f}  q95={float(dc.quantile(0.95)):.0f}  max={int(dc.max())}')
print(f'  fraction of genes with >= 10 clans: {(dc >= 10).sum() / dc.len() * 100:.2f}%')

print('\n--- Distinct clans per gene (CheckAMG, kept) ---')
dc = clans['checkamg_keep_clan_n']
print(f'  n={dc.len():,}  mean={float(dc.mean()):.2f}  median={float(dc.median()):.0f}  q95={float(dc.quantile(0.95)):.0f}  max={int(dc.max())}')
print(f'  fraction of genes with >= 10 clans: {(dc >= 10).sum() / dc.len() * 100:.2f}%')

print('\n--- Support for DRAM-V AMG calls ---')
hl = amg.group_by('support_class').len().with_columns(
    (pl.col('len') / amg.height).alias('frac')).sort('support_class')
for r in hl.iter_rows(named=True):
    print(f'  {r["support_class"]:18s} {r["len"]:>10,}  {r["frac"]*100:6.2f}%')
print(f'  total Pfam-based DRAM-V AMG calls: {amg.height:,}')

print('\n--- Support for DRAM-V AMG calls, by Pfam ---')
totals = []
fracs = []

for r in top_pfam.iter_rows(named=True):
    total = r['total']
    supported = r.get('hmmer_supported', 0) + r.get('hmmer_borderline', 0)
    frac_supported = (supported / total * 100) if total else 0

    totals.append(total)
    fracs.append(frac_supported)

totals = np.array(totals)
fracs = np.array(fracs)

print(f'Number of Pfams: {len(totals)}')

print(f'Calls per Pfam:')
print(f'  min:    {totals.min():>8,.0f}')
print(f'  max:    {totals.max():>8,.0f}')
print(f'  median: {np.median(totals):>8,.0f}')
print(f'  mean:   {totals.mean():>8,.2f}')

print(f'\n% support (supported + borderline):')
print(f'  min:    {fracs.min():>8.2f}%')
print(f'  max:    {fracs.max():>8.2f}%')
print(f'  median: {np.median(fracs):>8.2f}%')
print(f'  mean:   {fracs.mean():>8.2f}%')

print('\n--- Support for DRAM-V AMG calls, by Pfam (top 20) ---')

top20 = top_pfam.sort('total', descending=True).head(20)

totals = []
fracs = []

for r in top20.iter_rows(named=True):
    total = r['total']
    supported = r.get('hmmer_supported', 0) + r.get('hmmer_borderline', 0)
    frac_supported = (supported / total * 100) if total else 0

    totals.append(total)
    fracs.append(frac_supported)

totals = np.array(totals)
fracs = np.array(fracs)

print(f'Number of Pfams: {len(totals)}')

print(f'Calls per Pfam:')
print(f'  min:    {totals.min():>8,.0f}')
print(f'  max:    {totals.max():>8,.0f}')
print(f'  median: {np.median(totals):>8,.0f}')
print(f'  mean:   {totals.mean():>8,.2f}')

print(f'\n% support (supported + borderline):')
print(f'  min:    {fracs.min():>8.2f}%')
print(f'  max:    {fracs.max():>8.2f}%')
print(f'  median: {np.median(fracs):>8.2f}%')
print(f'  mean:   {fracs.mean():>8.2f}%')

--- Benchmark-wide per-gene Pfam hit counts ---
  DRAM-V           n=4,656,285  mean=267.83  median=290  q95=295  max=300
  CheckAMG (all)   n=4,656,285  mean=1.11    median=0    q95=4    max=116
  CheckAMG (kept)  n=4,656,285  mean=0.28    median=0    q95=1    max=47
  hmmscan          n=2,484,029  mean=4.16    median=2    q95=16   max=404
  hmmscan (cut_ga) n=2,484,029  mean=1.08    median=1    q95=4    max=58

--- Distinct clans per gene (DRAM-V) ---
  n=300,000  mean=212.60  median=227  q95=245  max=275
  fraction of genes with >= 10 clans: 99.94%

--- Distinct clans per gene (CheckAMG, all) ---
  n=300,000  mean=0.62  median=0  q95=2  max=65
  fraction of genes with >= 10 clans: 0.02%

--- Distinct clans per gene (CheckAMG, kept) ---
  n=300,000  mean=0.23  median=0  q95=1  max=3
  fraction of genes with >= 10 clans: 0.00%

--- Support for DRAM-V AMG calls ---


  hmmer_borderline       65,550    0.66%
  hmmer_rejected      9,764,367   97.89%
  hmmer_supported       144,446    1.45%
  total Pfam-based DRAM-V AMG calls: 9,974,363

--- Support for DRAM-V AMG calls, by Pfam ---
Number of Pfams: 257
Calls per Pfam:
  min:       2,797
  max:     128,045
  median:   33,347
  mean:   38,810.75

% support (supported + borderline):
  min:        0.00%
  max:       60.14%
  median:     1.09%
  mean:       3.36%

--- Support for DRAM-V AMG calls, by Pfam (top 20) ---
Number of Pfams: 20
Calls per Pfam:
  min:      77,888
  max:     128,045
  median:   88,080
  mean:   89,718.55

% support (supported + borderline):
  min:        0.01%
  max:        2.92%
  median:     0.23%
  mean:       0.53%
